# Healthcare Operations Intelligence

**IBM SkillsBuild Data Analytics with AI Internship 2026**  
**Student:** Nitin Kohli

This notebook converts the supplied Word project report into an **actual Jupyter/Python analytics workflow** rather than simply placing the report text into Markdown.

> **Important:** The supplied report describes a synthetic healthcare dataset. The notebook is for educational analytics only and is not for diagnosis, treatment, triage, or clinical decision-making.


## Project workflow

**Raw CSV → data-quality audit → cleaning → EDA → cohort analysis → Operations Load Index → synthetic leakage audit → ML benchmark → artifacts**

The report states that the raw dataset contains **150,500 rows and 14 columns**, and that cleaning produces **150,000 unique rows**.


In [ ]:
# 1. Imports and configuration

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mutual_info_score
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# Change this path if your CSV has a different name/location.
DATA_PATH = Path("data/healthcare_data.csv")

OUTPUT_DIR = Path("outputs")
CHART_DIR = OUTPUT_DIR / "charts"
ARTIFACT_DIR = Path("artifacts")
DATA_DIR = Path("data")

for folder in [OUTPUT_DIR, CHART_DIR, ARTIFACT_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 2. Load the raw healthcare dataset

In [ ]:
# Load raw data
# Expected columns from the report:
# patient_id, age, gender, symptoms, temperature, duration_days,
# bp_systolic, bp_diastolic, heart_rate, oxygen_saturation,
# pain_level, symptom_severity, chronic_condition, department

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place the healthcare CSV there or update DATA_PATH above."
    )

df_raw = pd.read_csv(DATA_PATH)

print("Raw shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())


## 3. Initial data-quality audit

In [ ]:
# Basic audit
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(3),
    "unique_count": df_raw.nunique(dropna=False),
})
display(audit)

print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Total missing cells:", int(df_raw.isna().sum().sum()))

# Numeric validity checks described in the report
if "oxygen_saturation" in df_raw:
    print("Oxygen saturation > 100:", int((df_raw["oxygen_saturation"] > 100).sum()))

if "bp_systolic" in df_raw:
    extreme_bp = ((df_raw["bp_systolic"] < 60) | (df_raw["bp_systolic"] > 200)).sum()
    print("Extreme systolic BP values:", int(extreme_bp))


## 4. Cleaning pipeline

In [ ]:
# Work on a copy; never overwrite the raw dataset.
df = df_raw.copy()

# Row-level audit flags
df["_had_duplicate"] = False
df["_had_missing"] = df.isna().any(axis=1)
df["_had_oxygen_correction"] = False
df["_had_bp_correction"] = False

# 1) Remove exact duplicates
duplicate_mask = df.duplicated(keep="first")
duplicate_count = int(duplicate_mask.sum())
df = df.loc[~duplicate_mask].copy()

# 2) Normalize categorical labels
categorical_cols = [
    c for c in ["gender", "symptoms", "symptom_severity", "chronic_condition", "department"]
    if c in df.columns
]

for col in categorical_cols:
    df[col] = df[col].astype("string").str.strip()

# Preserve ENT as an acronym and normalize common casing variations.
if "department" in df.columns:
    dept_map = {
        "ent": "ENT",
        "E.N.T.": "ENT",
        "e.n.t.": "ENT",
    }
    df["department"] = df["department"].replace(dept_map)

# 3) Correct oxygen saturation values above 100
if "oxygen_saturation" in df.columns:
    mask = df["oxygen_saturation"] > 100
    df.loc[mask, "oxygen_saturation"] = 100.0

# 4) Treat extreme systolic BP as missing
if "bp_systolic" in df.columns:
    mask = (df["bp_systolic"] < 60) | (df["bp_systolic"] > 200)
    df.loc[mask, "bp_systolic"] = np.nan

# 5) Numeric imputation with medians
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if not c.startswith("_")]

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# 6) Categorical imputation with modes
for col in categorical_cols:
    if df[col].isna().any():
        mode = df[col].mode(dropna=True)
        if len(mode):
            df[col] = df[col].fillna(mode.iloc[0])

# 7) Row-level final quality status
df["quality_status"] = np.where(
    df["_had_missing"], "required_correction", "clean"
)

# Re-check
print("Rows after duplicate removal:", len(df))
print("Remaining missing cells:", int(df.isna().sum().sum()))
print("Unique patient IDs:", df["patient_id"].nunique() if "patient_id" in df else "N/A")
display(df.head())


## 5. Cleaning summary

In [ ]:
quality_summary = pd.DataFrame({
    "metric": [
        "Raw rows",
        "Exact duplicates removed",
        "Raw missing cells",
        "Oxygen > 100",
        "Extreme systolic BP",
        "Final missing cells",
        "Final unique patient IDs",
    ],
    "value": [
        len(df_raw),
        duplicate_count,
        int(df_raw.isna().sum().sum()),
        int((df_raw["oxygen_saturation"] > 100).sum()) if "oxygen_saturation" in df_raw else np.nan,
        int(((df_raw["bp_systolic"] < 60) | (df_raw["bp_systolic"] > 200)).sum()) if "bp_systolic" in df_raw else np.nan,
        int(df.isna().sum().sum()),
        int(df["patient_id"].nunique()) if "patient_id" in df else np.nan,
    ],
})
display(quality_summary)

# Save cleaned/enriched dataset
clean_path = DATA_DIR / "healthcare_cleaned_150000.csv"
enriched_path = DATA_DIR / "healthcare_enriched_150000.csv"

df.drop(columns=[c for c in df.columns if c.startswith("_")], errors="ignore").to_csv(clean_path, index=False)
df.to_csv(enriched_path, index=False)

print("Saved:", clean_path)
print("Saved:", enriched_path)


## 6. Exploratory Data Analysis (EDA)

In [ ]:
# Core descriptive statistics
display(df.describe(include="all").T)

print("Average age:", round(df["age"].mean(), 2))
print("Median age:", round(df["age"].median(), 2))
print("Average heart rate:", round(df["heart_rate"].mean(), 2))

if "chronic_condition" in df:
    chronic_pct = (df["chronic_condition"].astype(str).str.lower().eq("yes").mean() * 100)
    print("Chronic-condition prevalence:", round(chronic_pct, 2), "%")


In [ ]:
# Age distribution
plt.figure(figsize=(10, 5))
sns.histplot(df["age"], bins=20, kde=True)
plt.title("Patient Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(CHART_DIR / "patient_age_distribution.png", dpi=150)
plt.show()


In [ ]:
# Department workload
department_counts = df["department"].value_counts()

plt.figure(figsize=(11, 5))
department_counts.sort_values(ascending=False).plot(kind="bar")
plt.title("Department Workload")
plt.xlabel("Department")
plt.ylabel("Patient Records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(CHART_DIR / "department_workload.png", dpi=150)
plt.show()

print("Busiest department:", department_counts.idxmax(), department_counts.max())


In [ ]:
# Symptom severity mix
severity_counts = df["symptom_severity"].value_counts()

plt.figure(figsize=(7, 5))
severity_counts.plot(kind="bar")
plt.title("Symptom Severity Mix")
plt.xlabel("Severity")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(CHART_DIR / "symptom_severity_mix.png", dpi=150)
plt.show()


In [ ]:
# Chronic-condition mix by age group
bins = [0, 20, 40, 60, 80, np.inf]
labels = ["0-20", "21-40", "41-60", "61-80", "81+"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)

chronic_by_age = pd.crosstab(
    df["age_group"],
    df["chronic_condition"],
)

display(chronic_by_age)

chronic_by_age.plot(kind="bar", figsize=(9, 5))
plt.title("Chronic-Condition Mix by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(CHART_DIR / "chronic_condition_by_age_group.png", dpi=150)
plt.show()


In [ ]:
# Top reported symptoms
top_symptoms = df["symptoms"].value_counts().head(10)

plt.figure(figsize=(11, 5))
top_symptoms.sort_values().plot(kind="barh")
plt.title("Top Reported Symptoms")
plt.xlabel("Count")
plt.ylabel("Symptom")
plt.tight_layout()
plt.savefig(CHART_DIR / "top_reported_symptoms.png", dpi=150)
plt.show()

print("Most common symptom:", top_symptoms.index[0], int(top_symptoms.iloc[0]))


## 7. Ten key questions from the report

In [ ]:
key_answers = {
    "Average patient age": round(df["age"].mean(), 2),
    "Most common gender": df["gender"].value_counts().idxmax(),
    "Most common symptom": df["symptoms"].value_counts().idxmax(),
    "Busiest department": df["department"].value_counts().idxmax(),
    "Most common severity": df["symptom_severity"].value_counts(normalize=True).idxmax(),
    "Average heart rate": round(df["heart_rate"].mean(), 2),
    "Highest average heart rate department": df.groupby("department")["heart_rate"].mean().idxmax(),
    "Chronic-condition prevalence (%)": round(
        df["chronic_condition"].astype(str).str.lower().eq("yes").mean() * 100, 2
    ),
    "Largest age group": df["age_group"].value_counts().idxmax(),
    "Highest average pain department": df.groupby("department")["pain_level"].mean().idxmax(),
}

for question, answer in key_answers.items():
    print(f"{question}: {answer}")


## 8. Department Operations Load Index

In [ ]:
# The report defines:
# volume share 40%, severe-case share 25%, chronic-condition share 15%,
# average symptom duration 20%.
#
# Each component is min-max normalized across departments.

dept = df.groupby("department").agg(
    patients=("department", "size"),
    severe_pct=("symptom_severity", lambda s: (s.astype(str).str.lower() == "severe").mean() * 100),
    chronic_pct=("chronic_condition", lambda s: (s.astype(str).str.lower() == "yes").mean() * 100),
    avg_duration=("duration_days", "mean"),
).reset_index()

dept["volume_pct"] = dept["patients"] / len(df) * 100

def minmax(series):
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - lo) / (hi - lo)

dept["volume_norm"] = minmax(dept["volume_pct"])
dept["severe_norm"] = minmax(dept["severe_pct"])
dept["chronic_norm"] = minmax(dept["chronic_pct"])
dept["duration_norm"] = minmax(dept["avg_duration"])

dept["ops_load"] = (
    0.40 * dept["volume_norm"]
    + 0.25 * dept["severe_norm"]
    + 0.15 * dept["chronic_norm"]
    + 0.20 * dept["duration_norm"]
) * 100

dept = dept.sort_values("ops_load", ascending=False)

display(dept[[
    "department", "patients", "volume_pct",
    "severe_pct", "chronic_pct", "avg_duration", "ops_load"
]])


In [ ]:
# Operations Load Index chart
plt.figure(figsize=(11, 5))
plt.bar(dept["department"], dept["ops_load"])
plt.title("Transparent Operations Load Index")
plt.xlabel("Department")
plt.ylabel("Operations Load Index")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(CHART_DIR / "operations_load_index.png", dpi=150)
plt.show()

dept.to_csv(ARTIFACT_DIR / "department_operations_load.csv", index=False)


In [ ]:
# Department profile across selected metrics
profile = dept.set_index("department")[[
    "volume_pct", "severe_pct", "chronic_pct", "avg_duration"
]]
display(profile)


## 9. Synthetic-data audit: symptom → department

In [ ]:
# Cramer's V implementation
def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape

    if n == 0:
        return np.nan

    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)

    denominator = min(kcorr - 1, rcorr - 1)
    return np.sqrt(phi2corr / denominator) if denominator > 0 else np.nan

cramers_v_score = cramers_v(df["symptoms"], df["department"])
print("Cramer's V (symptoms vs department):", round(cramers_v_score, 3))

# Simple symptom-to-most-common-department lookup
lookup = (
    df.groupby("symptoms")["department"]
      .agg(lambda s: s.mode().iloc[0])
)

pred_lookup = df["symptoms"].map(lookup)
lookup_accuracy = (pred_lookup == df["department"]).mean()

print("Symptom lookup accuracy:", round(lookup_accuracy * 100, 2), "%")

leakage_audit = {
    "cramers_v": float(cramers_v_score),
    "symptom_lookup_accuracy": float(lookup_accuracy),
}
(ARTIFACT_DIR / "synthetic_leakage_audit.json").write_text(
    json.dumps(leakage_audit, indent=2),
    encoding="utf-8"
)


### Methodological decision

The supplied report says the symptom feature has an unusually strong synthetic relationship with department. Therefore, **symptoms are excluded from the final ML benchmark** so the benchmark does not simply learn the near-hard-coded synthetic relationship.


## 10. Exploratory machine-learning benchmark

In [ ]:
# Target and features follow the report.
target = "department"

features = [
    "age",
    "temperature",
    "duration_days",
    "bp_systolic",
    "bp_diastolic",
    "heart_rate",
    "oxygen_saturation",
    "pain_level",
    "gender",
    "chronic_condition",
    "symptom_severity",
]

X = df[features].copy()
y = df[target].copy()

numeric_features = [
    "age", "temperature", "duration_days", "bp_systolic",
    "bp_diastolic", "heart_rate", "oxygen_saturation", "pain_level"
]
categorical_features = ["gender", "chronic_condition", "symptom_severity"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

models = {
    "Majority Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, pred),
        "Macro F1": f1_score(y_test, pred, average="macro"),
    })
    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df)


In [ ]:
# Benchmark visualization
plot_df = results_df.set_index("Model")[["Accuracy", "Balanced accuracy", "Macro F1"]]

plot_df.plot(kind="bar", figsize=(11, 6))
plt.title("Four-Model Exploratory Benchmark")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, max(0.30, plot_df.max().max() * 1.15))
plt.tight_layout()
plt.savefig(CHART_DIR / "four_model_benchmark.png", dpi=150)
plt.show()

results_df.to_csv(ARTIFACT_DIR / "model_benchmark.csv", index=False)


In [ ]:
# Detailed classification report and confusion matrix for Random Forest
rf_model = trained_models["Random Forest"]
rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, rf_pred, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, fmt="d", xticklabels=labels, yticklabels=labels)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Department")
plt.ylabel("Actual Department")
plt.tight_layout()
plt.savefig(CHART_DIR / "random_forest_confusion_matrix.png", dpi=150)
plt.show()


## 11. Evidence-based Q&A outputs

In [ ]:
# A small evidence table calculated directly from the cleaned dataset.
evidence = {
    "average_patient_age": float(df["age"].mean()),
    "busiest_department": str(df["department"].value_counts().idxmax()),
    "busiest_department_records": int(df["department"].value_counts().max()),
    "most_common_symptom": str(df["symptoms"].value_counts().idxmax()),
    "most_common_symptom_records": int(df["symptoms"].value_counts().max()),
    "average_heart_rate": float(df["heart_rate"].mean()),
    "chronic_condition_pct": float(
        df["chronic_condition"].astype(str).str.lower().eq("yes").mean() * 100
    ),
    "cramers_v_symptoms_department": float(cramers_v_score),
    "symptom_lookup_accuracy": float(lookup_accuracy),
}

display(pd.DataFrame([evidence]).T.rename(columns={0: "value"}))

(ARTIFACT_DIR / "evidence_qa.json").write_text(
    json.dumps(evidence, indent=2),
    encoding="utf-8"
)


## 12. Export final analytical artifacts

In [ ]:
# Save important outputs
dept.to_json(ARTIFACT_DIR / "department_operations_load.json", orient="records", indent=2)
results_df.to_json(ARTIFACT_DIR / "model_benchmark.json", orient="records", indent=2)

# Save a compact project summary
summary = {
    "raw_rows": int(len(df_raw)),
    "cleaned_rows": int(len(df)),
    "duplicate_rows_removed": int(duplicate_count),
    "final_missing_cells": int(df.isna().sum().sum()),
    "average_age": float(df["age"].mean()),
    "busiest_department": str(df["department"].value_counts().idxmax()),
    "cramers_v": float(cramers_v_score),
    "symptom_lookup_accuracy": float(lookup_accuracy),
    "model_benchmark": results_df.to_dict(orient="records"),
}

(ARTIFACT_DIR / "project_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("Artifacts generated successfully.")
print("Charts:", CHART_DIR)
print("Artifacts:", ARTIFACT_DIR)


## 13. Optional Streamlit dashboard code

The report specifies a Streamlit presentation layer with:

- Executive Overview
- Data Quality
- Department Intelligence
- Patient & Symptom Explorer
- ML Audit
- Evidence Q&A
- Downloads

The dashboard is normally kept in a separate `app.py`. The notebook above contains the complete analytical workflow that feeds that dashboard.


In [ ]:
# Optional: generate a starter app.py from the notebook's outputs.
# Run this cell only if you want the Streamlit dashboard file.

app_code = r'''
import json
from pathlib import Path

import pandas as pd
import streamlit as st

st.set_page_config(page_title="Healthcare Operations Intelligence", layout="wide")

DATA_PATH = Path("data/healthcare_enriched_150000.csv")
ARTIFACT_DIR = Path("artifacts")

st.title("Healthcare Operations Intelligence")
st.caption("Synthetic healthcare data — educational analytics only.")

if not DATA_PATH.exists():
    st.error(f"Dataset not found: {DATA_PATH}")
    st.stop()

df = pd.read_csv(DATA_PATH)

page = st.sidebar.selectbox(
    "Navigate",
    [
        "Executive Overview",
        "Data Quality",
        "Department Intelligence",
        "Patient & Symptom Explorer",
        "ML Audit",
        "Evidence Q&A",
        "Downloads",
    ],
)

if page == "Executive Overview":
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Records", f"{len(df):,}")
    c2.metric("Average Age", f"{df['age'].mean():.2f}")
    c3.metric("Busiest Department", df["department"].value_counts().idxmax())
    c4.metric("Chronic Condition", f"{df['chronic_condition'].astype(str).str.lower().eq('yes').mean()*100:.2f}%")

    st.subheader("Department Workload")
    st.bar_chart(df["department"].value_counts())

elif page == "Data Quality":
    st.subheader("Data Quality")
    st.write("Missing cells:", int(df.isna().sum().sum()))
    st.write("Duplicate rows:", int(df.duplicated().sum()))
    if "quality_status" in df:
        st.dataframe(df["quality_status"].value_counts())

elif page == "Department Intelligence":
    st.subheader("Department Intelligence")
    p = ARTIFACT_DIR / "department_operations_load.csv"
    if p.exists():
        st.dataframe(pd.read_csv(p), use_container_width=True)
    else:
        st.info("Run the notebook first to generate department artifacts.")

elif page == "Patient & Symptom Explorer":
    st.subheader("Patient & Symptom Explorer")
    departments = sorted(df["department"].dropna().unique())
    selected = st.multiselect("Department", departments, default=departments)
    filtered = df[df["department"].isin(selected)]
    st.write("Records:", len(filtered))
    st.dataframe(filtered.head(100), use_container_width=True)

elif page == "ML Audit":
    st.subheader("ML Audit")
    p = ARTIFACT_DIR / "model_benchmark.csv"
    if p.exists():
        st.dataframe(pd.read_csv(p), use_container_width=True)
    leakage = ARTIFACT_DIR / "synthetic_leakage_audit.json"
    if leakage.exists():
        st.json(json.loads(leakage.read_text()))

elif page == "Evidence Q&A":
    st.subheader("Evidence Q&A")
    p = ARTIFACT_DIR / "evidence_qa.json"
    if p.exists():
        st.json(json.loads(p.read_text()))
    else:
        st.info("Run the notebook first.")

elif page == "Downloads":
    st.subheader("Downloads")
    for p in [
        Path("data/healthcare_cleaned_150000.csv"),
        Path("data/healthcare_enriched_150000.csv"),
        ARTIFACT_DIR / "model_benchmark.csv",
        ARTIFACT_DIR / "department_operations_load.csv",
    ]:
        if p.exists():
            st.download_button(
                label=f"Download {p.name}",
                data=p.read_bytes(),
                file_name=p.name,
            )
'''

Path("app.py").write_text(app_code, encoding="utf-8")
print("Generated app.py")


## 14. Limitations

The source report states that the dataset is synthetic, has no time dimension, contains an unusually strong symptom-to-department relationship, uses median/mode imputation, and is not a clinical prediction system. The Operations Load Index is also a custom descriptive score rather than a medical risk or severity score.

The notebook preserves those limitations rather than presenting the analysis as real-world clinical evidence.


## 15. Run order

1. Put the source healthcare CSV at `data/healthcare_data.csv` (or change `DATA_PATH`).
2. Run cells from top to bottom.
3. The notebook creates cleaned/enriched CSV files, charts and JSON/CSV artifacts.
4. Run the optional `app.py` generation cell if you want the Streamlit dashboard.
5. Start the dashboard with:

```bash
streamlit run app.py
```
